In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install sentencepiece
!pip install -U bitsandbytes
!pip install peft
!pip install accelerate
!pip install -q -U datasets scipy ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 57.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 52.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.optimization import get_cosine_with_hard_restarts_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=8
    weight_decay=1e-6
    max_len = 60
    fold = 5

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/df_test.csv', encoding = 'utf-8')
df_train_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_train_pre.csv', encoding = 'utf-8')
df_dev_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_dev_pre.csv', encoding = 'utf-8')
df_test_pre = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/데이터전처리/df_test_pre.csv', encoding = 'utf-8')

df_train_pre.dropna(inplace = True)
df_dev_pre.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train_pre.reset_index(drop = True, inplace = True)
df_dev_pre.reset_index(drop = True, inplace = True)

cuda


In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id = "beomi/llama-2-ko-7b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config, num_labels = args.label_size)

config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00015.safetensors:   0%|          | 0.00/919M [00:00<?, ?B/s]

model-00002-of-00015.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

model-00003-of-00015.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

...

model-00013-of-00015.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

model-00014-of-00015.safetensors:   0%|          | 0.00/742M [00:00<?, ?B/s]

model-00015-of-00015.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at beomi/llama-2-ko-7b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    model_max_length=args.max_len,
    add_eos_token=True,
    padding=True,
    truncation=True)

tokenizer.eos_token = tokenizer.eos_token
tokenizer.pad_token = tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/842 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
from tokenizers.processors import TemplateProcessing

class CustomDataset(Dataset):
    def __init__(self, dataset, text, label, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.label = label
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        label = self.dataset.iloc[idx]['output']

        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding=True,
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask, label

    def __len__(self):
        return len(self.dataset)

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch])  # 레이블을 Tensor로 변환

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks, labels

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

dir = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold0.pt'
torch.save(peft_model.state_dict(), dir)

In [ ]:
import transformers
from datetime import datetime
import torch.optim as optim
import torch.nn.functional as F

peft_model.config.pad_token_id = tokenizer.pad_token_id

kf = KFold(n_splits=5, shuffle=True, random_state=42)

df = pd.concat([df_train_pre, df_dev_pre])

# KFold 교차 검증
fold = 0
for fold, (train_index, valid_index) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}/{args.fold}")

    peft_model.load_state_dict(torch.load(dir))
    peft_model.to(device)

    df_train = df.iloc[train_index].reset_index(drop=True)
    df_valid = df.iloc[valid_index].reset_index(drop=True)

    train_dataset = CustomDataset(df_train, 'input', 'output', tokenizer, args.max_len)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_batch)
    valid_dataset = CustomDataset(df_valid, 'input', 'output', tokenizer, args.max_len)
    valid_loader = DataLoader(valid_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = AdamW(peft_model.parameters(), lr=args.start_lr)
    scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(optimizer,
                                                                num_warmup_steps = int(len(train_loader)*args.epochs * 0.1),
                                                                num_training_steps = len(train_loader)*args.epochs)

    stop_val_f1 = 0
    stop_count = 0

    for epoch in range(args.epochs):
        if stop_count == 4:
            break
        peft_model.train()
        train_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(train_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            outputs = peft_model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()
            scheduler.step()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(train_loss / len(pbar), 4)))

        train_loss = train_loss / len(train_loader)
        train_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        train_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        peft_model.eval()
        valid_loss = 0
        target_lst = []
        pred_lst = []
        pbar = tqdm(valid_loader)
        for ids, mask, target in pbar:
            ids = ids.to(device)
            mask = mask.to(device)
            target = target.to(device)

            outputs = peft_model(ids, mask)
            loss = loss_fn(outputs.logits, target).to(device)
            valid_loss += loss.item()

            target_lst.extend(target.detach().cpu().numpy())
            pred_lst.extend(outputs.logits.argmax(dim = 1).tolist())
            pbar.set_description('\033[1m[C_loss : {:>.5}]\033[0m'.format(round(valid_loss / len(pbar), 4)))
        valid_loss = valid_loss / len(valid_loader)
        valid_score = f1_score(y_true=target_lst, y_pred=pred_lst, average='micro')
        valid_acc = accuracy_score(y_true=target_lst, y_pred=pred_lst)

        torch.cuda.empty_cache()
        gc.collect()

        print(f'Fold {fold + 1}, Epoch : {epoch + 1},    t_loss : {round(train_loss, 4)},   t_f1 : {round(train_score, 4)},   t_acc : {round(train_acc, 4)}\n')
        print(f'                     v_loss : {round(valid_loss, 4)},   v_f1 : {round(valid_score, 4)},   v_acc : {round(valid_acc, 4)}\n')

        if valid_score > stop_val_f1:
            default_weight_path = path + f'llama2_ko_7b_fold{fold + 1}.pt'
            torch.save(peft_model.state_dict(), default_weight_path)
            stop_val_f1 = valid_score
            stop_count = 0
        else:
            stop_count += 1

        torch.cuda.empty_cache()
        gc.collect()

Fold 1/5


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/1850 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.2403]: 100%|██████████| 463/463 [02:36<00:00,  2.97it/s]


Fold 1, Epoch : 1,    t_loss : 0.5845,   t_f1 : 0.7192,   t_acc : 0.7192

                     v_loss : 0.2403,   v_f1 : 0.9084,   v_acc : 0.9084



  0%|          | 0/1850 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.1852]: 100%|██████████| 463/463 [02:36<00:00,  2.96it/s]


Fold 1, Epoch : 2,    t_loss : 0.193,   t_f1 : 0.9265,   t_acc : 0.9265

                     v_loss : 0.1852,   v_f1 : 0.9289,   v_acc : 0.9289



...

Fold 5, Epoch : 7,    t_loss : 0.0176,   t_f1 : 0.9957,   t_acc : 0.9957

                     v_loss : 0.3385,   v_f1 : 0.9346,   v_acc : 0.9346



  0%|          | 0/1850 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.3861]: 100%|██████████| 463/463 [02:37<00:00,  2.94it/s]


Fold 5, Epoch : 8,    t_loss : 0.0096,   t_f1 : 0.9979,   t_acc : 0.9979

                     v_loss : 0.3861,   v_f1 : 0.9319,   v_acc : 0.9319



  0%|          | 0/1850 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
[C_loss : 0.4305]: 100%|██████████| 463/463 [02:37<00:00,  2.94it/s]


Fold 5, Epoch : 9,    t_loss : 0.0061,   t_f1 : 0.9984,   t_acc : 0.9984

                     v_loss : 0.4305,   v_f1 : 0.9321,   v_acc : 0.9321



In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id = "beomi/llama-2-ko-7b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForSequenceClassification.from_pretrained(base_model_id, quantization_config=bnb_config)


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so
CUDA SETUP: Highest compute capability among GPUs detected: 8.0
CUDA SETUP: Detected CUDA version 118
CUDA SETUP: Loading binary /usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cuda118.so...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: /usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('http'), PosixPath('//172.28.0.1'), PosixPath('8013')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('--logtostderr --listen_host=172.28.0.12 --

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at beomi/llama-2-ko-7b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    model_max_length=args.max_len,
    add_eos_token=True,
    padding=True,
    truncation=True)

tokenizer.eos_token = tokenizer.eos_token
tokenizer.pad_token = tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token

def collate_batch(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]

    # 패딩 추가
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0)

    return input_ids, attention_masks

test_dataset = TestDataset(df_test_pre, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05, bias="none"
)

model = prepare_model_for_kbit_training(base_model)
peft_model = get_peft_model(model, peft_config)

In [ ]:
peft_model.config.pad_token_id = tokenizer.pad_token_id

model_paths = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold1.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold2.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold3.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold4.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/llama2_ko_7b_fold5.pt']

models = []
for model_path in model_paths:
    peft_model.load_state_dict(torch.load(model_path))
    peft_model.to(device)
    peft_model.eval()
    models.append(model)

res = []
with torch.no_grad():
    for ids, mask in tqdm(test_loader):
        ids = ids.to(device)
        mask = mask.to(device)

        outputs = [peft_model(ids, mask).logits for model in models]
        avg_output = sum(outputs) / len(models)
        res.extend(avg_output.argmax(axis=1).tolist())

df_test['output'] = res

100%|██████████| 259/259 [12:52<00:00,  2.98s/it]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",0
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test28__kfold_llama2_ko_7b_new.jsonl')

In [ ]:
torch.cuda.empty_cache()
gc.collect()

26

In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id1 = "beomi/llama-2-ko-7b"
base_model_id2 = "EleutherAI/polyglot-ko-12.8b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

tokenizer1 = AutoTokenizer.from_pretrained(
    base_model_id1,
    model_max_length=100,
    padding_side="right",
    add_eos_token=True)

tokenizer1.eos_token = tokenizer1.eos_token
tokenizer1.pad_token = tokenizer1.pad_token if tokenizer1.pad_token is not None else tokenizer1.eos_token

test_dataset1 = TestDataset(df_test, 'input', tokenizer1, args.max_len)
test_loader1 = DataLoader(test_dataset1, batch_size=args.batch_size, shuffle=False)

tokenizer2 = AutoTokenizer.from_pretrained(
    base_model_id2,
    model_max_length=100,
    padding_side="right",
    add_eos_token=True)

tokenizer2.eos_token = tokenizer2.eos_token
tokenizer2.pad_token = tokenizer2.pad_token if tokenizer2.pad_token is not None else tokenizer2.eos_token

test_dataset2 = TestDataset(df_test, 'input', tokenizer2, args.max_len)
test_loader2 = DataLoader(test_dataset2, batch_size=args.batch_size, shuffle=False)

model_name = "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(model_name)
test_dataset = TestDataset(df_test, 'input', tokenizer, args.max_len)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
)
model1 = AutoModelForSequenceClassification.from_pretrained(base_model_id1, quantization_config = bnb_config)
model1 = get_peft_model(model1, peft_config)

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at beomi/llama-2-ko-7b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05
)

model2 = AutoModelForSequenceClassification.from_pretrained(base_model_id2, quantization_config = bnb_config)
model2 = get_peft_model(model2, peft_config)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/52.5k [00:00<?, ?B/s]

model-00001-of-00028.safetensors:   0%|          | 0.00/946M [00:00<?, ?B/s]

model-00002-of-00028.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

model-00003-of-00028.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

...

model-00006-of-00028.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

model-00007-of-00028.safetensors:   0%|          | 0.00/896M [00:00<?, ?B/s]

model-00008-of-00028.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

model-00009-of-00028.safetensors:   0%|          | 0.00/896M [00:00<?, ?B/s]

model-00010-of-00028.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

In [ ]:
model1.config.pad_token_id = tokenizer1.pad_token_id
model2.config.pad_token_id = tokenizer2.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

model_paths1 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_llama2_ko_7b_kfold.pt',
               ]
model_paths2 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_polyglot_ko_12.8b_kfold.pt'
               ]
model_paths = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_kc_electra.pt'
               ]

models = []
models1 = []
models2 = []

# 모델1 로드
for model_path in model_paths1:
    model1.load_state_dict(torch.load(model_path))
    model1.to(device)
    model1.eval()
    models1.append(model1)

# 모델2 로드
for model_path in model_paths2:
    model2.load_state_dict(torch.load(model_path))
    model2.to(device)
    model2.eval()
    models2.append(model2)

for model_path in model_paths:
    model3 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=args.label_size)
    model3.load_state_dict(torch.load(model_path))
    model3.to(device)
    model3.eval()
    models.append(model3)

res = []
with torch.no_grad():
    for i, (data1, data2, data3) in enumerate(tqdm(zip(test_loader1, test_loader2, test_loader))):
        ids1, mask1 = data1[0].to(device), data1[1].to(device)
        ids2, mask2 = data2[0].to(device), data2[1].to(device)
        ids3, mask3 = data3[0].to(device), data3[1].to(device)

        outputs1 = [model(input_ids=ids1, attention_mask=mask1)[0] for model in models1]
        outputs2 = [model(input_ids=ids2, attention_mask=mask2)[0] for model in models2]
        outputs3 = [model(input_ids=ids3, attention_mask=mask3)[0] for model in models]
        outputs = outputs1 + outputs2 + outputs3

        avg_output = sum(outputs) / len(outputs)
        avg_output = avg_output.cpu().numpy()

        res.extend(avg_output.argmax(axis=1).tolist())

        torch.cuda.empty_cache()
        gc.collect()

df_test['output'] = res

In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/test21_polyglot_12.8b_llama2_ko_7b.jsonl')